Straßen Graph

In [1]:
import osmium
import csv
import pickle
import math

KEEP_HIGHWAY = {
    "motorway", "motorway_link",
    "trunk", "trunk_link",
    "primary", "primary_link",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link",
    "unclassified",
    "residential"
}

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000.0

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c


class EdgeGeoHandler(osmium.SimpleHandler):
    def __init__(self, writer):
        super().__init__()
        self.writer = writer
        self.arc_id = 0

        # ✅ Kompakte Graph-Struktur
        self.nodes_used = set()
        self.arcs = []              # nur (arc_id, u, v)
        self.Dist = {}
        self.highway = {}
        self.name = {}
        self.tunnel = {}

        # ✅ NUR EINMAL Koordinaten speichern
        self.node_coords = {}

        # Stats
        self.total_candidate_segments = 0
        self.written_segments = 0
        self.skipped_invalid_location = 0


    def _write_arc(self, u, v, way_id, hwy, oneway, name,
                   lat1, lon1, lat2, lon2, tunnel):

        dist = haversine(lat1, lon1, lat2, lon2)

        # CSV (optional)
        self.writer.writerow([
            self.arc_id, u, v, way_id, hwy, oneway, name,
            lat1, lon1, lat2, lon2, dist, tunnel
        ])

        self.nodes_used.add(u)
        self.nodes_used.add(v)

        # ✅ Node-Koordinaten (NUR EINMAL speichern)
        if u not in self.node_coords:
            self.node_coords[u] = (lat1, lon1)
        if v not in self.node_coords:
            self.node_coords[v] = (lat2, lon2)

        # ✅ Kompakt speichern
        self.arcs.append((self.arc_id, u, v))

        self.Dist[self.arc_id] = dist
        self.highway[self.arc_id] = hwy
        self.name[self.arc_id] = name
        self.tunnel[self.arc_id] = tunnel

        self.arc_id += 1
        self.written_segments += 1


    def way(self, w):
        hwy = w.tags.get("highway")
        if hwy not in KEEP_HIGHWAY:
            return

        if len(w.nodes) < 2:
            return

        oneway = w.tags.get("oneway", "no")
        name = w.tags.get("name", "")
        tunnel = w.tags.get("tunnel", "no")

        node_list = list(w.nodes)

        for n1, n2 in zip(node_list[:-1], node_list[1:]):
            self.total_candidate_segments += 1

            try:
                lat1 = n1.location.lat
                lon1 = n1.location.lon
                lat2 = n2.location.lat
                lon2 = n2.location.lon
            except Exception:
                self.skipped_invalid_location += 1
                continue

            u = n1.ref
            v = n2.ref

            # forward
            self._write_arc(u, v, w.id, hwy, oneway, name,
                           lat1, lon1, lat2, lon2, tunnel)

            # reverse (falls nicht Einbahnstraße)
            if oneway in ("", "no", "false", "0"):
                self._write_arc(v, u, w.id, hwy, oneway, name,
                               lat2, lon2, lat1, lon1, tunnel)


def build_graph_with_coords(
    pbf_path,
    edges_out="edges_germany_geo.csv",
    pkl_out="germany_graph_geo_optimized.pkl"
):

    with open(edges_out, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        writer.writerow([
            "arc_id", "u", "v", "way_id", "highway", "oneway", "name",
            "lat1", "lon1", "lat2", "lon2", "dist", "tunnel"
        ])

        handler = EdgeGeoHandler(writer)
        handler.apply_file(pbf_path, locations=True)

    # ✅ Kompakte PKL
    data = {
        "nodes": list(handler.nodes_used),
        "arcs": handler.arcs,
        "Dist": handler.Dist,
        "node_coords": handler.node_coords,
        "highway": handler.highway,   # OPTIONAL
        "name": handler.name,         # OPTIONAL (Straßennamen)
        "tunnel": handler.tunnel      # OPTIONAL
    }

    with open(pkl_out, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

    print("✅ Fertig!")
    print("✔ Nodes:", len(data["nodes"]))
    print("✔ Arcs:", len(data["arcs"]))
    print("✔ Kandidaten:", handler.total_candidate_segments)
    print("✔ Geschrieben:", handler.written_segments)
    print("✔ Übersprungen:", handler.skipped_invalid_location)


if __name__ == "__main__":
    pbf_path = r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\germany-latest.osm.pbf"

    build_graph_with_coords(
        pbf_path,
        edges_out="edges_germany_geo_optimized.csv",
        pkl_out="germany_geo_optimized.pkl"
    )

✅ Fertig!
✔ Nodes: 23812224
✔ Arcs: 46941353
✔ Kandidaten: 24677546
✔ Geschrieben: 46941353
✔ Übersprungen: 0


In [2]:
build_graph_with_coords(
    pbf_path,
    edges_out=r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\edges_germnay_geo_optimized.csv",
    pkl_out=r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\edges_germany_geo_optimized.pkl"
)


✅ Fertig!
✔ Nodes: 23812224
✔ Arcs: 46941353
✔ Kandidaten: 24677546
✔ Geschrieben: 46941353
✔ Übersprungen: 0


Unfalldichte

In [1]:
import pickle
from pathlib import Path
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString

# ---------------------------------------------------
# 1. Graph laden
# ---------------------------------------------------
datei = Path.home() / "sciebo" / "OperationsResearch" / "berlin_graph_geo_com.pkl"

with open(datei, "rb") as f:
    pulp_data = pickle.load(f)

print("Keys im Graphen:", pulp_data.keys())

arcs = pulp_data["arcs"]
node_coords = pulp_data["node_coords"]

print("✅ Graph geladen")
print(f"Knoten: {len(node_coords)}")
print(f"Kanten: {len(arcs)}")

# ---------------------------------------------------
# 2. Straßen-Geometrien aus arcs + node_coords bauen
# ---------------------------------------------------
road_rows = []

for arc in arcs:
    arc_id = arc["arc_id"]
    u = arc["u"]
    v = arc["v"]

    if u in node_coords and v in node_coords:
        lat1, lon1 = node_coords[u]
        lat2, lon2 = node_coords[v]

        geom = LineString([
            (lon1, lat1),   # WICHTIG: (lon, lat)
            (lon2, lat2)
        ])

        road_rows.append({
            "arc_id": arc_id,
            "u": u,
            "v": v,
            "geometry": geom
        })

gdf_roads = gpd.GeoDataFrame(road_rows, crs="EPSG:4326").to_crs(epsg=3857)

print("✅ Straßen-Geometrie erstellt:", len(gdf_roads))

# ---------------------------------------------------
# 3. Unfalldaten laden
# ---------------------------------------------------
csv_path = r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\Umwelftfreundliche_Routenplanung\data\raw\Accidents.csv"

df = pd.read_csv(csv_path, sep=";", low_memory=False)
df.columns = df.columns.str.strip()

# Koordinaten fixen (Komma -> Punkt)
df["XGCSWGS84"] = df["XGCSWGS84"].astype(str).str.replace(",", ".", regex=False)
df["YGCSWGS84"] = df["YGCSWGS84"].astype(str).str.replace(",", ".", regex=False)

df["XGCSWGS84"] = pd.to_numeric(df["XGCSWGS84"], errors="coerce")
df["YGCSWGS84"] = pd.to_numeric(df["YGCSWGS84"], errors="coerce")

df = df.dropna(subset=["XGCSWGS84", "YGCSWGS84"]).copy()

# Optional: Nur bestimmtes Bundesland
# Beispiel Berlin: df = df[df["ULAND"] == 11].copy()
# Für ganz Deutschland: KEIN Filter

# Gewicht schon VOR dem Join definieren
df["weight"] = df["UKATEGORIE"].map({
    1: 5,
    2: 3,
    3: 1
}).fillna(1)

# Unfallpunkte als GeoDataFrame
# WICHTIG: X = lon, Y = lat
gdf_acc = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["XGCSWGS84"], df["YGCSWGS84"]),
    crs="EPSG:4326"
).to_crs(epsg=3857)

print("✅ Unfälle geladen:", len(gdf_acc))

# ---------------------------------------------------
# 4. Unfälle auf nächste Kante mappen
# ---------------------------------------------------
joined = gpd.sjoin_nearest(
    gdf_acc,
    gdf_roads,
    how="left",
    max_distance=50,      # je nach Datenqualität evtl. 30 / 50 / 100 testen
    distance_col="dist_to_road"
)

print("✅ Matching fertig")
print("Gematchte Unfälle:", joined["arc_id"].notna().sum())

# Nur erfolgreiche Matches behalten
joined_valid = joined.dropna(subset=["arc_id"]).copy()
joined_valid["arc_id"] = joined_valid["arc_id"].astype(int)

# ---------------------------------------------------
# 5. Unfallzahlen pro Kante berechnen
# ---------------------------------------------------
acc_counts = joined_valid.groupby("arc_id").size()
weighted_counts = joined_valid.groupby("arc_id")["weight"].sum()

# In Straßen-GDF zurückschreiben
gdf_roads["accidents"] = gdf_roads["arc_id"].map(acc_counts).fillna(0)
gdf_roads["weighted_accidents"] = gdf_roads["arc_id"].map(weighted_counts).fillna(0)

# Länge in Metern
gdf_roads["length"] = gdf_roads.geometry.length

# Unfälle pro Meter
gdf_roads["score"] = gdf_roads["accidents"] / gdf_roads["length"]

# gewichtete Unfälle pro Meter
gdf_roads["weighted_score"] = gdf_roads["weighted_accidents"] / gdf_roads["length"]

# NaN / inf absichern
gdf_roads["score"] = gdf_roads["score"].fillna(0)
gdf_roads["weighted_score"] = gdf_roads["weighted_score"].fillna(0)

print("✅ Scores berechnet")

# ---------------------------------------------------
# 6. Edge-DataFrame für Export
# ---------------------------------------------------
df_edges = gdf_roads[[
    "arc_id", "u", "v", "length", "accidents", "weighted_accidents",
    "score", "weighted_score"
]].copy()

# Distanz aus PKL ergänzen, falls vorhanden
if "Dist" in pulp_data:
    df_edges["distance"] = df_edges["arc_id"].map(pulp_data["Dist"])
else:
    df_edges["distance"] = df_edges["length"]

# Highway / Name ergänzen, falls vorhanden
if "highway" in pulp_data:
    df_edges["highway"] = df_edges["arc_id"].map(pulp_data["highway"])

if "name" in pulp_data:
    df_edges["name"] = df_edges["arc_id"].map(pulp_data["name"])

print("✅ Edge-Daten erstellt:", df_edges.shape)

# ---------------------------------------------------
# 7. Node-Unfallzuordnung (optional)
# ---------------------------------------------------
node_rows = []

for node, coord in node_coords.items():
    lat, lon = coord
    node_rows.append({
        "node": node,
        "lat": lat,
        "lon": lon
    })

df_nodes = pd.DataFrame(node_rows)

gdf_nodes = gpd.GeoDataFrame(
    df_nodes,
    geometry=gpd.points_from_xy(df_nodes["lon"], df_nodes["lat"]),
    crs="EPSG:4326"
).to_crs(epsg=3857)

joined_nodes = gpd.sjoin_nearest(
    gdf_acc,
    gdf_nodes,
    how="left",
    max_distance=30,
    distance_col="dist_to_node"
)

joined_nodes_valid = joined_nodes.dropna(subset=["node"]).copy()

node_counts = joined_nodes_valid.groupby("node").size()
node_weighted = joined_nodes_valid.groupby("node")["weight"].sum()

df_nodes["accidents"] = df_nodes["node"].map(node_counts).fillna(0)
df_nodes["weighted_accidents"] = df_nodes["node"].map(node_weighted).fillna(0)

print("✅ Unfälle auf Knoten gemappt")

# ---------------------------------------------------
# 8. Export
# ---------------------------------------------------
export_data = {
    "edges": df_edges,
    "nodes": df_nodes,
    "matched_accidents": joined_valid
}

output_path = r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\berlin_graph_with_accidents_test_v1.pkl"

with open(output_path, "wb") as f:
    pickle.dump(export_data, f, protocol=pickle.HIGHEST_PROTOCOL)

print("✅ Export fertig:", output_path)

c:\Users\tspol\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\tspol\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Keys im Graphen: dict_keys(['nodes', 'arcs', 'cost', 'highway', 'name', 'coords', 'Dist', 'node_coords', 'tunnel'])
✅ Graph geladen
Knoten: 294724
Kanten: 533453
✅ Straßen-Geometrie erstellt: 533453
✅ Unfälle geladen: 268519
✅ Matching fertig
Gematchte Unfälle: 15198
✅ Scores berechnet
✅ Edge-Daten erstellt: (533453, 11)
✅ Unfälle auf Knoten gemappt
✅ Export fertig: C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\berlin_graph_with_accidents_test_v1.pkl


Naturschutzgebiete

In [8]:
import pickle
from pathlib import Path

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString

# =========================================================
# 1. PFade
# =========================================================
datei = Path.home() / "sciebo" / "OperationsResearch" / "germany_graph_geo.pkl"

nature_path = Path(
    r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\Umwelftfreundliche_Routenplanung\data\processed\nature_reserves_1.gpkg"
)

output_path = Path(
    r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\germany_graph_with_nature.pkl"
)


output_path.parent.mkdir(parents=True, exist_ok=True)

# =========================================================
# 2. DEUTSCHLAND-GRAPH LADEN
# =========================================================
with open(datei, "rb") as f:
    pulp_data = pickle.load(f)

print("Keys im Graph:", pulp_data.keys())

arcs = pulp_data["arcs"]
node_coords = pulp_data["node_coords"]

print("✅ Graph geladen")
print(f"Anzahl Knoten: {len(node_coords)}")
print(f"Anzahl Kanten: {len(arcs)}")

# Optional vorhandene Zusatzinfos
dist_map = pulp_data.get("Dist", {})
highway_map = pulp_data.get("highway", {})
name_map = pulp_data.get("name", {})

# =========================================================
# 3. OPTIONALER KOORDINATEN-CHECK
# =========================================================
# Deutschland-Koordinaten typischerweise:
# lon ~ 6 bis 15
# lat ~ 47 bis 55
# Falls stattdessen (lat, lon) vorliegt, drehen wir um.

sample_items = list(node_coords.items())[:20]
xs = [coord[0] for _, coord in sample_items]
ys = [coord[1] for _, coord in sample_items]

if pd.Series(xs).mean() > 40 and pd.Series(ys).mean() < 20:
    swap_xy = True
    print("⚠️ Koordinaten wirken wie (lat, lon) -> werden für Geometrie in (lon, lat) umgedreht")
else:
    swap_xy = False
    print("✅ Koordinaten wirken wie (lon, lat) oder plausibel für direkte Nutzung")


def split_coord(coord):
    a, b = coord
    if swap_xy:
        # input = (lat, lon)
        lat, lon = a, b
        return lat, lon
    else:
        # input = (lon, lat)
        lon, lat = a, b
        return lat, lon


def make_point(coord):
    lat, lon = split_coord(coord)
    return Point(lon, lat)


# =========================================================
# 4. NODES ALS GEODATAFRAME
# =========================================================
node_rows = []
node_geoms = []

for node_id, coord in node_coords.items():
    try:
        lat, lon = split_coord(coord)
        pt = Point(lon, lat)

        node_rows.append({
            "node": node_id,
            "lat": lat,
            "lon": lon
        })
        node_geoms.append(pt)
    except Exception:
        continue

gdf_nodes = gpd.GeoDataFrame(node_rows, geometry=node_geoms, crs="EPSG:4326")

print("✅ Node-Geometrien erstellt:", len(gdf_nodes))

# =========================================================
# 5. EDGES / STRASSEN ALS GEODATAFRAME
# =========================================================
edge_rows = []
edge_geoms = []

skipped_edges = 0

for arc in arcs:
    arc_id = arc["arc_id"]
    u = arc["u"]
    v = arc["v"]

    if u in node_coords and v in node_coords:
        try:
            p1 = make_point(node_coords[u])
            p2 = make_point(node_coords[v])

            edge_rows.append({
                "arc_id": arc_id,
                "u": u,
                "v": v,
                "distance_m": dist_map.get(arc_id, None),
                "highway": highway_map.get(arc_id, ""),
                "name": name_map.get(arc_id, "")
            })
            edge_geoms.append(LineString([p1, p2]))

        except Exception:
            skipped_edges += 1
    else:
        skipped_edges += 1

gdf_edges = gpd.GeoDataFrame(edge_rows, geometry=edge_geoms, crs="EPSG:4326")

print("✅ Edge-Geometrien erstellt")
print(f"⚠️ Übersprungene Kanten: {skipped_edges}")
print(f"✅ Verwendete Kanten: {len(gdf_edges)}")

# =========================================================
# 6. NATURSCHUTZGEBIETE LADEN
# =========================================================
gdf_nature = gpd.read_file(nature_path)

print("✅ Naturschutzgebiete geladen")
print(f"Anzahl Gebiete: {len(gdf_nature)}")
print("CRS Naturflächen:", gdf_nature.crs)

# Nur gültige Geometrien behalten
gdf_nature = gdf_nature[~gdf_nature.geometry.isna()].copy()
gdf_nature = gdf_nature[gdf_nature.geometry.is_valid].copy()

# Falls NAME fehlt, sauber absichern
if "NAME" not in gdf_nature.columns:
    gdf_nature["NAME"] = ""

# =========================================================
# 7. GEMEINSAMES METRISCHES CRS
# =========================================================
# Für Deutschland/Europa sinnvoll in Metern:
target_crs = "EPSG:3035"

gdf_nodes = gdf_nodes.to_crs(target_crs)
gdf_edges = gdf_edges.to_crs(target_crs)
gdf_nature = gdf_nature.to_crs(target_crs)

print("✅ Knoten, Kanten und Naturflächen ins gleiche metrische CRS transformiert")

# =========================================================
# 8. KNOTEN -> NATURSCHUTZGEBIETE MAPPEN
# =========================================================

# 8a) Liegt Knoten innerhalb eines Naturschutzgebiets?
node_join = gpd.sjoin(
    gdf_nodes,
    gdf_nature[["NAME", "geometry"]],
    how="left",
    predicate="within"
)

node_agg = (
    node_join.groupby("node")
    .agg(
        in_nature=("NAME", lambda x: x.notna().any()),
        reserve_count=("NAME", lambda x: x.dropna().nunique()),
        reserve_names=("NAME", lambda x: "; ".join(sorted(set(x.dropna().astype(str)))) if x.notna().any() else "")
    )
    .reset_index()
)

gdf_nodes = gdf_nodes.merge(node_agg, on="node", how="left")

gdf_nodes["in_nature"] = gdf_nodes["in_nature"].fillna(False)
gdf_nodes["reserve_count"] = gdf_nodes["reserve_count"].fillna(0).astype(int)
gdf_nodes["reserve_names"] = gdf_nodes["reserve_names"].fillna("")

# 8b) Distanz zum nächsten Naturschutzgebiet
node_nearest = gpd.sjoin_nearest(
    gdf_nodes[["node", "geometry"]],
    gdf_nature[["NAME", "geometry"]],
    how="left",
    distance_col="dist_to_nature_m"
)[["node", "dist_to_nature_m"]]

node_nearest = node_nearest.groupby("node", as_index=False)["dist_to_nature_m"].min()

gdf_nodes = gdf_nodes.drop(columns=["dist_to_nature_m"], errors="ignore")
gdf_nodes = gdf_nodes.merge(node_nearest, on="node", how="left")

gdf_nodes.loc[gdf_nodes["in_nature"], "dist_to_nature_m"] = 0.0

print("✅ Knoten auf Naturschutzgebiete gemappt")

# =========================================================
# 9. KANTEN -> NATURSCHUTZGEBIETE MAPPEN
# =========================================================

# Straßenlänge in Meter
gdf_edges["length_m"] = gdf_edges.geometry.length

# ---------------------------------------------------------
# 9a) Welche Kanten schneiden Naturschutzgebiete?
# ---------------------------------------------------------
# Overlay kann bei Deutschland groß werden.
# Daher zuerst nur intersectende Kanten per sjoin identifizieren.

edge_hits = gpd.sjoin(
    gdf_edges[["arc_id", "u", "v", "geometry", "length_m"]],
    gdf_nature[["NAME", "geometry"]],
    how="inner",
    predicate="intersects"
)

if len(edge_hits) > 0:
    # Nur die wirklich treffenden Kanten für exakte Intersection nehmen
    hit_ids = edge_hits["arc_id"].dropna().unique().tolist()

    gdf_edges_hit = gdf_edges[gdf_edges["arc_id"].isin(hit_ids)][
        ["arc_id", "u", "v", "geometry", "length_m"]
    ].copy()

    edge_intersection = gpd.overlay(
        gdf_edges_hit,
        gdf_nature[["NAME", "geometry"]],
        how="intersection"
    )

    if len(edge_intersection) > 0:
        edge_intersection["inside_length_m"] = edge_intersection.geometry.length

        edge_agg = (
            edge_intersection.groupby("arc_id")
            .agg(
                intersects_nature=("NAME", lambda x: True),
                reserve_count=("NAME", lambda x: x.dropna().nunique()),
                reserve_names=("NAME", lambda x: "; ".join(sorted(set(x.dropna().astype(str)))) if x.notna().any() else ""),
                inside_length_m=("inside_length_m", "sum")
            )
            .reset_index()
        )
    else:
        edge_agg = pd.DataFrame(columns=[
            "arc_id", "intersects_nature", "reserve_count", "reserve_names", "inside_length_m"
        ])
else:
    edge_agg = pd.DataFrame(columns=[
        "arc_id", "intersects_nature", "reserve_count", "reserve_names", "inside_length_m"
    ])

gdf_edges = gdf_edges.merge(edge_agg, on="arc_id", how="left")

gdf_edges["intersects_nature"] = gdf_edges["intersects_nature"].fillna(False)
gdf_edges["reserve_count"] = gdf_edges["reserve_count"].fillna(0).astype(int)
gdf_edges["reserve_names"] = gdf_edges["reserve_names"].fillna("")
gdf_edges["inside_length_m"] = gdf_edges["inside_length_m"].fillna(0.0)

# Anteil innerhalb von Naturschutzgebieten
gdf_edges["in_nature_ratio"] = 0.0
mask_nonzero = gdf_edges["length_m"] > 0
gdf_edges.loc[mask_nonzero, "in_nature_ratio"] = (
    gdf_edges.loc[mask_nonzero, "inside_length_m"] / gdf_edges.loc[mask_nonzero, "length_m"]
)

# ---------------------------------------------------------
# 9b) Distanz zur nächsten Naturfläche
# ---------------------------------------------------------
edge_nearest = gpd.sjoin_nearest(
    gdf_edges[["arc_id", "geometry"]],
    gdf_nature[["NAME", "geometry"]],
    how="left",
    distance_col="dist_to_nature_m"
)[["arc_id", "dist_to_nature_m"]]

edge_nearest = edge_nearest.groupby("arc_id", as_index=False)["dist_to_nature_m"].min()

gdf_edges = gdf_edges.drop(columns=["dist_to_nature_m"], errors="ignore")
gdf_edges = gdf_edges.merge(edge_nearest, on="arc_id", how="left")

# Wenn Straße Naturgebiet schneidet -> Distanz = 0
gdf_edges.loc[gdf_edges["intersects_nature"], "dist_to_nature_m"] = 0.0

# Optionaler Nature Score:
# hoch, wenn nahe/innerhalb Naturschutzgebiet
gdf_edges["nature_score"] = 1 / (gdf_edges["dist_to_nature_m"] + 1)

print("✅ Kanten auf Naturschutzgebiete gemappt")

# =========================================================
# 10. EXPORT-TABELLEN BAUEN
# =========================================================

# Knoten zurück nach WGS84 für Export-Koordinaten
gdf_nodes_wgs = gdf_nodes.to_crs(epsg=4326).copy()
gdf_nodes_wgs["x"] = gdf_nodes_wgs.geometry.x
gdf_nodes_wgs["y"] = gdf_nodes_wgs.geometry.y

df_nodes_export = gdf_nodes_wgs[[
    "node",
    "in_nature",
    "reserve_count",
    "reserve_names",
    "dist_to_nature_m",
    "x",
    "y"
]].copy()

# Kanten-Export
df_edges_export = gdf_edges[[
    "arc_id",
    "u",
    "v",
    "distance_m",
    "highway",
    "name",
    "length_m",
    "intersects_nature",
    "reserve_count",
    "reserve_names",
    "inside_length_m",
    "in_nature_ratio",
    "dist_to_nature_m",
    "nature_score"
]].copy()

print("✅ Export-Tabellen erstellt")
print("\nNode-Vorschau:")
print(df_nodes_export.head())

print("\nEdge-Vorschau:")
print(df_edges_export.head())

# =========================================================
# 11. ALS PKL SPEICHERN
# =========================================================
export_data = {
    "nodes": df_nodes_export,
    "edges": df_edges_export,
    "meta": {
        "source_graph": str(datei),
        "source_nature": str(nature_path),
        "target_crs": target_crs,
        "node_count": len(df_nodes_export),
        "edge_count": len(df_edges_export)
    }
}

with open(output_path, "wb") as f:
    pickle.dump(export_data, f)

print(f"\n✅ PKL gespeichert unter:\n{output_path}")


Keys im Graph: dict_keys(['nodes', 'arcs', 'cost', 'highway', 'name', 'coords', 'Dist', 'node_coords'])
✅ Graph geladen
Anzahl Knoten: 2020869
Anzahl Kanten: 3025675
⚠️ Koordinaten wirken wie (lat, lon) -> werden für Geometrie in (lon, lat) umgedreht
✅ Node-Geometrien erstellt: 2020869
✅ Edge-Geometrien erstellt
⚠️ Übersprungene Kanten: 0
✅ Verwendete Kanten: 3025675
✅ Naturschutzgebiete geladen
Anzahl Gebiete: 9012
CRS Naturflächen: EPSG:25832
✅ Knoten, Kanten und Naturflächen ins gleiche metrische CRS transformiert
✅ Knoten auf Naturschutzgebiete gemappt
✅ Kanten auf Naturschutzgebiete gemappt
✅ Export-Tabellen erstellt

Node-Vorschau:
         node  in_nature  reserve_count reserve_names  dist_to_nature_m  \
0  3786023396      False              0                     1633.476709   
1  3786023162      False              0                     1524.686895   
2    10210552      False              0                      884.349830   
3   277783046      False              0               

Populationsdichte

In [11]:
import pickle
from pathlib import Path
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString

# =========================================================
# 1. PFADE
# =========================================================
graph_path = Path.home() / "sciebo" / "OperationsResearch" / "germany_graph_geo.pkl"

pop_path = Path(
    r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\Umwelftfreundliche_Routenplanung\data\raw\ESTAT_Census_2021_V2.gpkg"
)

output_path = Path(
    r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\germany_graph_with_population.pkl"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

# =========================================================
# 2. GRAPH LADEN
# =========================================================
with open(graph_path, "rb") as f:
    pulp_data = pickle.load(f)

arcs = pulp_data["arcs"]
node_coords = pulp_data["node_coords"]
dist_map = pulp_data.get("Dist", {})

print("✅ Graph geladen")
print(f"Knoten: {len(node_coords)}")
print(f"Kanten: {len(arcs)}")

# =========================================================
# 3. KOORDINATEN HANDLING
# =========================================================
def split_coord(coord):
    a, b = coord
    # heuristisch
    if a > 40:
        return a, b   # (lat, lon)
    else:
        return b, a   # (lon, lat)

def make_point(coord):
    lat, lon = split_coord(coord)
    return Point(lon, lat)

# =========================================================
# 4. GEODATEN ERSTELLEN
# =========================================================
# Knoten
node_rows = []
node_geoms = []

for node, coord in node_coords.items():
    lat, lon = split_coord(coord)
    node_rows.append({"node": node, "lat": lat, "lon": lon})
    node_geoms.append(Point(lon, lat))

gdf_nodes = gpd.GeoDataFrame(node_rows, geometry=node_geoms, crs="EPSG:4326")

# Kanten
edge_rows = []
edge_geoms = []

for arc in arcs:
    u = arc["u"]
    v = arc["v"]
    arc_id = arc["arc_id"]

    if u in node_coords and v in node_coords:
        p1 = make_point(node_coords[u])
        p2 = make_point(node_coords[v])

        edge_rows.append({
            "arc_id": arc_id,
            "u": u,
            "v": v,
            "distance_m": dist_map.get(arc_id, None)
        })
        edge_geoms.append(LineString([p1, p2]))

gdf_edges = gpd.GeoDataFrame(edge_rows, geometry=edge_geoms, crs="EPSG:4326")

print("✅ Geometrien erstellt:", len(gdf_edges))

# =========================================================
# 5. BEVÖLKERUNG LADEN
# =========================================================
gdf_pop = gpd.read_file(pop_path)

print("✅ Bevölkerung geladen:", len(gdf_pop))

gdf_pop = gdf_pop[["T", "geometry"]]
gdf_pop = gdf_pop[gdf_pop["T"] > 0]

# =========================================================
# 6. CRS → metrisch!
# =========================================================
target_crs = "EPSG:3035"   # Europa -> Meter

gdf_edges = gdf_edges.to_crs(target_crs)
gdf_nodes = gdf_nodes.to_crs(target_crs)
gdf_pop = gdf_pop.to_crs(target_crs)

print("✅ CRS transformiert")

# =========================================================
# 7. KANTENLÄNGE
# =========================================================
gdf_edges["length_m"] = gdf_edges.geometry.length

# =========================================================
# 8. POPULATION MAPPING (BESSERER ANSATZ)
# =========================================================

# 👉 Buffer um Straße (z. B. 50 m)
BUFFER = 50

gdf_edges["geometry_buffer"] = gdf_edges.geometry.buffer(BUFFER)

gdf_edges_buffer = gdf_edges[["arc_id", "geometry_buffer"]].copy()
gdf_edges_buffer = gdf_edges_buffer.set_geometry("geometry_buffer")

# 👇 WICHTIG
gdf_edges_buffer = gdf_edges_buffer.set_geometry("geometry_buffer")

edge_pop = gpd.overlay(
    gdf_edges_buffer,
    gdf_pop,
    how="intersection"
)


# Bevölkerung aggregieren
if len(edge_pop) > 0:
    edge_pop_agg = (
        edge_pop.groupby("arc_id")
        .agg(population=("T", "sum"))
        .reset_index()
    )
else:
    edge_pop_agg = pd.DataFrame(columns=["arc_id", "population"])

# Merge zurück
gdf_edges = gdf_edges.merge(edge_pop_agg, on="arc_id", how="left")

gdf_edges["population"] = gdf_edges["population"].fillna(0)

# =========================================================
# 9. POPULATION PRO METER
# =========================================================
gdf_edges["pop_per_meter"] = gdf_edges["population"] / gdf_edges["length_m"]
gdf_edges["pop_per_meter"] = gdf_edges["pop_per_meter"].fillna(0)

print("✅ Bevölkerung gemappt")

# =========================================================
# 10. EXPORT
# =========================================================

df_nodes_export = gdf_nodes.to_crs(epsg=4326)[["node", "lat", "lon"]]

df_edges_export = gdf_edges[[
    "arc_id",
    "u",
    "v",
    "distance_m",
    "length_m",
    "population",
    "pop_per_meter"
]]

# speichern
export_data = {
    "nodes": df_nodes_export,
    "edges": df_edges_export
}

with open(output_path, "wb") as f:
    pickle.dump(export_data, f)

print("✅ Fertig:", output_path)


✅ Graph geladen
Knoten: 2020869
Kanten: 3025675
✅ Geometrien erstellt: 3025675
✅ Bevölkerung geladen: 4595749
✅ CRS transformiert
✅ Bevölkerung gemappt
✅ Fertig: C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\germany_graph_with_population.pkl
